In [0]:
# MAGIC %md
# MAGIC ## Silver Layer Transformation and Target Sink Writer
# MAGIC This notebook reads Bronze data from S3, writes it to the target Silver table defined in the config tables, and records batch metrics.

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

# ── Define Widgets ─────────────────────────────────────────────────────────
dbutils.widgets.text("config_id", "", "Config Ingestion Object ID")
dbutils.widgets.text("table_name", "", "Target Table Name")
dbutils.widgets.text("config_master_id", "", "Config Master ID")
dbutils.widgets.text("source_system_id", "", "Source System ID")
dbutils.widgets.text("landing_volume_path", "", "Base S3 / Landing Path")

config_id = int(dbutils.widgets.get("config_id"))
table_name = dbutils.widgets.get("table_name")
config_master_id = int(dbutils.widgets.get("config_master_id"))
source_system_id = int(dbutils.widgets.get("source_system_id"))
landing_volume_path = dbutils.widgets.get("landing_volume_path")

start_time = datetime.utcnow()
print(f"[SILVER] Starting Silver process for config_id={config_id} at {start_time}")

In [0]:
# MAGIC %md
# MAGIC ### 1. Resolve Child Configuration Table and Fetch Configurations

In [0]:
# ── Dynamic Child Configuration Table Resolution ────────────────────────────
CONFIG_MASTER_TABLE = "migration_x_catalog.pfl_x_schema.config_master"

master_rows = (
    spark.table(CONFIG_MASTER_TABLE)
    .filter(f"config_id = {config_master_id}")
    .collect()
)

if not master_rows:
    raise ValueError(f"No entry found in {CONFIG_MASTER_TABLE} for config_id={config_master_id}")

m_row = master_rows[0].asDict()
catalog = m_row.get("config_catalog_name")
schema = m_row.get("config_schema_name")
table = m_row.get("config_table_name")

child_table_fqn = f"{catalog}.{schema}.{table}"
print(f"[SILVER] Dynamically resolved child config table FQN: {child_table_fqn}")

# ── Fetch Config Object Row ──────────────────────────────────────────────────
child_df = spark.table(child_table_fqn).filter(f"Config_ID = {config_id} OR config_id = {config_id}")
config_row = child_df.collect()[0].asDict()

# Dynamic column resolution to support both casing/aliases
def get_val(keys, default=None):
    for k in keys:
        if k in config_row:
            return config_row[k]
    return default

source_name = get_val(["source_name", "Source_Name", "source_system_name"])
source_schema = get_val(["source_schema", "Source_Schema_Name"])
source_object_name = get_val(["source_object_name", "Source_Table_Name", "Source_Collection_Name"])
sink_schema_name = get_val(["Sink_Schema_Name", "target_schema"])
sink_table_name = get_val(["Sink_Table_Name", "target_table"])
target_catalog = get_val(["target_catalog"], "hive_metastore")
file_format = get_val(["file_format"], "parquet")
write_mode = get_val(["write_mode"], "overwrite")

print(f"Parsed configuration details:")
print(f"  Source: {source_name} | {source_schema}.{source_object_name}")
print(f"  Sink  : {target_catalog}.{sink_schema_name}.{sink_table_name}")
print(f"  Format: {file_format} | Mode: {write_mode}")

In [0]:
# MAGIC %md
# MAGIC ### 2. Read Bronze Data from S3 Landing Path

In [0]:
schema_part = f"{source_schema}/" if source_schema else ""
s3_path = f"{landing_volume_path.rstrip('/')}/{source_name}/{schema_part}{source_object_name}"
print(f"[SILVER] Reading raw bronze data from: {s3_path}")

# Load files dynamically from partition directories
df = spark.read.format(file_format).load(s3_path)
print(f"Loaded DataFrame row count: {df.count()}")

In [0]:
# MAGIC %md
# MAGIC ### 3. Write to Silver Medallion Layer Delta Target

In [0]:
# Ensure target schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{sink_schema_name}")

target_fqn = f"{target_catalog}.{sink_schema_name}.{sink_table_name}"
print(f"[SILVER] Writing Silver data to Delta table: {target_fqn}")

df.write.format("delta").mode(write_mode).option("mergeSchema", "true").saveAsTable(target_fqn)
print(f"[SILVER] Successfully wrote table: {target_fqn}")

In [0]:
# MAGIC %md
# MAGIC ### 4. Update Configuration Execution Dates

In [0]:
end_time = datetime.utcnow()

# Resolve case-sensitive column names of the child configuration table
columns = spark.table(child_table_fqn).columns

config_id_col = next((c for c in columns if c.lower() == "config_id"), "Config_ID")
last_sink_col = next((c for c in columns if c.lower() == "silver_sink_last_sink_date"), "silver_sink_last_sink_date")
started_col   = next((c for c in columns if c.lower() == "sink_batch_started_date"), "sink_batch_started_date")

print(f"Updating metadata dates in child table: {child_table_fqn}")
print(f"  Start: {start_time}")
print(f"  End: {end_time}")

spark.sql(f"""
    UPDATE {child_table_fqn}
    SET {last_sink_col} = '{end_time}',
        {started_col}   = '{start_time}'
    WHERE {config_id_col} = {config_id}
""")
print(f"[SILVER] Metadata update completed successfully.")